# 📦 Sistem Analisis ATK Menggunakan Algoritma K-Means
---
**Tujuan:** Mengelompokkan divisi dan produk ATK berdasarkan pola konsumsi menggunakan K-Means Clustering

**Library:** pandas, numpy, matplotlib, sklearn

**Normalisasi:** RobustScaler — tahan terhadap outlier, menggunakan Median dan IQR

---

In [ ]:
# ============================================================
#  IMPORT LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, silhouette_samples
from sklearn.preprocessing import RobustScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 140)

print('Semua library berhasil diimport!')
print("RobustScaler: X' = (X - Median) / IQR  [tahan outlier]")

In [ ]:
# ============================================================
#  FUNGSI UTILITAS
# ============================================================

# ----------------------------------------------------------
# 1. NORMALISASI ROBUST SCALER
# ----------------------------------------------------------
def normalisasi_robust(df, kolom):
    """
    RobustScaler: X' = (X - Median) / IQR   IQR = Q3 - Q1
    Tahan terhadap outlier.
    """
    df_norm = df.copy()
    stats = {}
    for col in kolom:
        q1     = df[col].quantile(0.25)
        median = df[col].median()
        q3     = df[col].quantile(0.75)
        iqr    = q3 - q1
        df_norm[col + '_norm'] = (df[col] - median) / iqr if iqr != 0 else 0.0
        stats[col] = {'Q1': q1, 'Median': median, 'Q3': q3, 'IQR': iqr}
    return df_norm, stats


# ----------------------------------------------------------
# 2. TABEL DENGAN ELLIPSIS (truncated display)
# ----------------------------------------------------------
def tampilkan_tabel_ellipsis(df, n_head=5, n_tail=3, label=''):
    """
    Tampilkan df dengan pola: n_head baris pertama ... n_tail baris terakhir.
    Jika total baris <= n_head+n_tail, tampilkan semua.
    """
    n = len(df)
    if label:
        print(f'  {label}')
    if n <= n_head + n_tail:
        print(df.round(4).to_string())
    else:
        print(df.head(n_head).round(4).to_string())
        print(f'  ... [{n - n_head - n_tail} baris tengah disembunyikan] ...')
        print(df.tail(n_tail).round(4).to_string())
    print(f'  Total baris: {n}')


# ----------------------------------------------------------
# 3. K-MEANS MANUAL
# ----------------------------------------------------------
def kmeans_manual(X, k, max_iter=100, random_state=42):
    np.random.seed(random_state)
    n = X.shape[0]
    idx = np.random.choice(n, k, replace=False)
    centroids = X[idx].copy()

    print('=' * 65)
    print(f'  INISIALISASI CENTROID AWAL (K={k})')
    print('=' * 65)
    df_init = pd.DataFrame(centroids, columns=[f'F{i+1}' for i in range(X.shape[1])])
    df_init.index = [f'C{i+1}' for i in range(k)]
    print(df_init.round(4).to_string())
    print()

    history = []
    labels  = np.zeros(n, dtype=int)

    for it in range(1, max_iter + 1):
        distances   = np.array([[np.sqrt(np.sum((x - c)**2)) for c in centroids] for x in X])
        new_labels  = np.argmin(distances, axis=1)
        new_centroids = np.array([
            X[new_labels == j].mean(axis=0) if np.any(new_labels == j) else centroids[j]
            for j in range(k)
        ])
        changed        = not np.array_equal(labels, new_labels)
        centroid_shift = np.max(np.abs(new_centroids - centroids))

        history.append({
            'iterasi'        : it,
            'distances'      : distances.copy(),
            'labels'         : new_labels.copy(),
            'centroids'      : new_centroids.copy(),   # centroid BARU (setelah update)
            'centroids_used' : centroids.copy(),       # centroid LAMA (yang dipakai hitung jarak)
            'centroid_shift' : centroid_shift,
            'changed'        : changed,
            'is_converged'   : False
        })

        labels    = new_labels.copy()
        centroids = new_centroids.copy()

        if not changed or centroid_shift < 1e-6:
            history[-1]['is_converged'] = True
            print(f'  Konvergen pada Iterasi {it}  '
                  f'(pergeseran centroid maks = {centroid_shift:.8f})')
            break

    return labels, centroids, history


# ----------------------------------------------------------
# 4. TAMPILKAN ITERASI (fitur + jarak + cluster + ellipsis)
# ----------------------------------------------------------
def tampilkan_iterasi(history, X, kolom_fitur, nama_baris, k, max_tampil_awal=3,
                      n_head=5, n_tail=3):
    """
    Tampilkan iterasi pertama (max_tampil_awal) DAN iterasi konvergen.
    Tabel: nama | fitur_1 .. fitur_n | Jarak_C1 .. Jarak_Ck | Cluster
    Data banyak ditampilkan dengan ellipsis.
    """
    total    = len(history)
    conv_idx = next((i for i, r in enumerate(history) if r['is_converged']), total - 1)
    awal_idxs = list(range(min(max_tampil_awal, total)))

    # Nama kolom fitur yang ditampilkan (pendek)
    feat_labels = [str(f)[:12] for f in kolom_fitur]

    def _cetak_iterasi(rec):
        it  = rec['iterasi']
        tag = '  [KONVERGEN]' if rec['is_converged'] else ''
        print('\n' + '-' * 75)
        print(f'  ITERASI {it}{tag}')
        print('-' * 75)

        # -- Contoh perhitungan jarak step-by-step (2 data pertama, iterasi 1 saja) --
        if it == 1:
            print('  [Contoh Perhitungan Jarak Euclidean - 2 Data Pertama]')
            print(f'  Rumus: d(x, C) = sqrt( sum_fi (x_fi - C_fi)^2 )')
            print()
            # Gunakan centroids_used = centroid SEBELUM update
            # yang benar-benar dipakai saat menghitung jarak iterasi ini
            cents_used = rec.get('centroids_used', rec['centroids'])
            for row_idx in range(min(2, len(X))):
                xi    = X[row_idx]
                label = nama_baris[row_idx]
                print(f'  >> Data: {label}')
                print(f'     Nilai fitur : {dict(zip(feat_labels, [round(v,4) for v in xi]))}')
                for j in range(k):
                    cj       = cents_used[j]   # centroid LAMA (dipakai hitung jarak)
                    komponen = [(f, round(float(xi[fi]),4), round(float(cj[fi]),4),
                                 round(float((xi[fi]-cj[fi])**2), 6))
                                for fi, f in enumerate(feat_labels)]
                    sum_sq   = sum(v[3] for v in komponen)
                    jarak    = sum_sq ** 0.5
                    # Verifikasi: bandingkan dengan jarak di tabel
                    jarak_tabel = float(rec['distances'][row_idx][j])
                    detail   = "  +  ".join(
                        f"({v[1]}-{v[2]})^2={v[3]}" for v in komponen
                    )
                    print(f'     Jarak ke C{j+1}: sqrt({detail})')
                    print(f'                  = sqrt({round(sum_sq,6)}) = {round(jarak,4)}'
                          f'  [tabel: {round(jarak_tabel,4)}]')
                assign = int(np.argmin(rec["distances"][row_idx]))
                print(f'     -> Masuk ke C{assign+1} (jarak terkecil)')
                print()

        # Bangun DataFrame: kolom fitur | Jarak_Cx | Cluster
        feat_data  = {feat_labels[fi]: X[:, fi] for fi in range(X.shape[1])}
        dist_data  = {f'Jarak_C{j+1}': rec['distances'][:, j] for j in range(k)}
        clust_data = {'Cluster': [f'C{l+1}' for l in rec['labels']]}

        df_all = pd.DataFrame({**feat_data, **dist_data, **clust_data},
                               index=nama_baris)

        print('  [Tabel Lengkap: Nilai Fitur | Jarak ke Centroid | Assignment Cluster]')
        n = len(df_all)
        if n <= n_head + n_tail:
            print(df_all.round(4).to_string())
        else:
            print(df_all.head(n_head).round(4).to_string())
            print(f'  ... [{n - n_head - n_tail} baris tengah disembunyikan] ...')
            print(df_all.tail(n_tail).round(4).to_string())
        print(f'  Total: {n} baris')

        # Centroid baru
        df_cent = pd.DataFrame(
            rec['centroids'],
            columns=feat_labels,
            index=[f'C{j+1}' for j in range(k)]
        )
        print(f'\n  [Update Centroid Baru - Iterasi {it}]')
        print(df_cent.round(4).to_string())
        print(f'  Pergeseran Centroid Maks = {rec["centroid_shift"]:.8f}')

    for i in awal_idxs:
        _cetak_iterasi(history[i])

    if conv_idx not in awal_idxs:
        skip_dari = history[awal_idxs[-1]]['iterasi'] + 1
        skip_ke   = history[conv_idx]['iterasi'] - 1
        if skip_dari <= skip_ke:
            print(f'\n  ... iterasi {skip_dari} s/d {skip_ke} diringkas '
                  f'(pola assignment tidak berubah signifikan) ...')
        _cetak_iterasi(history[conv_idx])


# ----------------------------------------------------------
# 5. WCSS DETAIL
# ----------------------------------------------------------
def hitung_wcss_detail(X, labels, centroids, nama_baris=None, n_head=5, n_tail=3):
    k = len(centroids)
    n = X.shape[0]
    if nama_baris is None:
        nama_baris = [f'Data_{i}' for i in range(n)]

    print('  [Rumus WCSS]')
    print('  WCSS = Sum_cluster Sum_{i in cluster} ||x_i - mu_cluster||^2')
    print()

    rows = []
    wcss_total = 0
    cluster_ss = {}

    for j in range(k):
        mask    = labels == j
        members = np.where(mask)[0]
        ss_cl   = 0
        for idx in members:
            sq = np.sum((X[idx] - centroids[j])**2)
            ss_cl += sq
            rows.append({'Nama': nama_baris[idx], 'Cluster': f'C{j+1}', '||x-mu||^2': sq})
        cluster_ss[f'C{j+1}'] = {'n_anggota': int(np.sum(mask)), 'SS_Cluster': ss_cl}
        wcss_total += ss_cl

    df_wcss = pd.DataFrame(rows)
    tampilkan_tabel_ellipsis(df_wcss, n_head=n_head, n_tail=n_tail,
                              label='[Tabel ||x_i - mu||^2 per Data Point]')

    print()
    print('  [SS per Cluster]')
    df_ss = pd.DataFrame(cluster_ss).T.reset_index()
    df_ss.columns = ['Cluster','n_anggota','SS_Cluster']
    print(df_ss.round(6).to_string(index=False))
    print(f'\n  WCSS Total = {wcss_total:.6f}')
    return wcss_total, cluster_ss


# ----------------------------------------------------------
# 6. SILHOUETTE DETAIL
# ----------------------------------------------------------
def hitung_silhouette_detail(X, labels, nama_baris=None, n_head=5, n_tail=3):
    n = X.shape[0]
    if nama_baris is None:
        nama_baris = [f'Data_{i}' for i in range(n)]
    unique_labels = np.unique(labels)

    print('  [Rumus Silhouette Score]')
    print('  a(i) = rata-rata jarak titik i ke semua titik dalam cluster-nya')
    print('  b(i) = rata-rata jarak terkecil titik i ke cluster lain terdekat')
    print('  s(i) = (b(i) - a(i)) / max(a(i), b(i))')
    print('  Silhouette Score = mean(s(i))')
    print()

    sil_vals = silhouette_samples(X, labels)
    a_vals   = np.zeros(n)
    b_vals   = np.zeros(n)
    for i in range(n):
        ci   = labels[i]
        same = np.where(labels == ci)[0]
        same = same[same != i]
        a_vals[i] = np.mean([np.sqrt(np.sum((X[i]-X[j])**2)) for j in same]) if len(same) > 0 else 0.0
        b_cands = []
        for cj in unique_labels:
            if cj == ci:
                continue
            other = np.where(labels == cj)[0]
            b_cands.append(np.mean([np.sqrt(np.sum((X[i]-X[j])**2)) for j in other]))
        b_vals[i] = min(b_cands) if b_cands else 0.0

    rows = [{'Nama': nama_baris[i], 'Cluster': f'C{labels[i]+1}',
             'a(i)': a_vals[i], 'b(i)': b_vals[i], 's(i)': sil_vals[i]}
            for i in range(n)]
    df_sil = pd.DataFrame(rows)
    tampilkan_tabel_ellipsis(df_sil, n_head=n_head, n_tail=n_tail,
                              label='[Tabel a(i), b(i), s(i) per Data Point]')

    sil_mean = sil_vals.mean()
    print()
    print('  [Rata-rata s(i) per Cluster]')
    for lbl in unique_labels:
        mask = labels == lbl
        print(f'  C{lbl+1}: mean s(i) = {sil_vals[mask].mean():.4f}  (n={np.sum(mask)})')
    print(f'\n  Silhouette Score = mean(s(i)) = {sil_mean:.4f}')
    return sil_mean


# ----------------------------------------------------------
# 7. DBI DETAIL
# ----------------------------------------------------------
def hitung_dbi_detail(X, labels, centroids):
    k = len(centroids)
    print('  [Rumus Davies-Bouldin Index]')
    print('  s_i  = rata-rata jarak titik cluster i ke centroid i')
    print('  d_ij = jarak Euclidean antara centroid i dan j')
    print('  R_ij = (s_i + s_j) / d_ij')
    print('  D_i  = max_{j!=i} R_ij')
    print('  DBI  = (1/k) x Sigma D_i')
    print()

    s = np.zeros(k)
    for j in range(k):
        mask = labels == j
        if np.any(mask):
            s[j] = np.mean([np.sqrt(np.sum((X[i]-centroids[j])**2)) for i in np.where(mask)[0]])

    print('  [s_i - Rata-rata Jarak Intra-Cluster ke Centroid]')
    for j in range(k):
        print(f'  s(C{j+1}) = {s[j]:.4f}')

    print()
    print('  [d_ij - Jarak Antar Centroid]')
    d = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            d[i,j] = np.sqrt(np.sum((centroids[i]-centroids[j])**2))
    df_d = pd.DataFrame(d, index=[f'C{i+1}' for i in range(k)],
                            columns=[f'C{j+1}' for j in range(k)])
    print(df_d.round(4).to_string())

    print()
    print('  [R_ij = (s_i + s_j) / d_ij]')
    R = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if i != j and d[i,j] > 0:
                R[i,j] = (s[i]+s[j]) / d[i,j]
    df_R = pd.DataFrame(R, index=[f'C{i+1}' for i in range(k)],
                            columns=[f'C{j+1}' for j in range(k)])
    print(df_R.round(4).to_string())

    D = np.array([max(R[i,j] for j in range(k) if j!=i) for i in range(k)])
    print()
    print('  [D_i = max_{j!=i} R_ij]')
    for i in range(k):
        print(f'  D(C{i+1}) = {D[i]:.4f}')
    dbi = np.mean(D)
    print(f'\n  DBI = mean(D_i) = {dbi:.4f}')
    return dbi


# ----------------------------------------------------------
# 8. EVALUASI LENGKAP
# ----------------------------------------------------------
def evaluasi_kmeans(X, labels, centroids, k, nama_baris=None):
    print(f'\n' + '='*65)
    print(f'  EVALUASI LENGKAP - K={k}')
    print('='*65)

    print('\n' + '-'*65)
    print('  4.1  WITHIN-CLUSTER SUM OF SQUARES (WCSS)')
    print('-'*65)
    wcss, _ = hitung_wcss_detail(X, labels, centroids, nama_baris)

    print('\n' + '-'*65)
    print('  4.2  SILHOUETTE SCORE')
    print('-'*65)
    sil = hitung_silhouette_detail(X, labels, nama_baris)

    print('\n' + '-'*65)
    print('  4.3  DAVIES-BOULDIN INDEX (DBI)')
    print('-'*65)
    dbi = hitung_dbi_detail(X, labels, centroids)

    print(f'\n' + '='*65)
    print(f'  RINGKASAN EVALUASI K={k}')
    print(f'  WCSS             = {wcss:.4f}')
    print(f'  Silhouette Score = {sil:.4f}  (makin tinggi makin baik, max=1)')
    print(f'  DBI              = {dbi:.4f}  (makin rendah makin baik, min=0)')
    print('='*65)
    return {'K': k, 'WCSS': wcss, 'Silhouette': sil, 'DBI': dbi}


# ----------------------------------------------------------
# 9. ELBOW METHOD
# ----------------------------------------------------------
def elbow_method(X, k_range, judul='Elbow Method'):
    wcss_list = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X)
        wcss_list.append(km.inertia_)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(list(k_range), wcss_list, 'bo-', linewidth=2, markersize=8)
    for k, w in zip(k_range, wcss_list):
        ax.annotate(f'{w:.2f}', (k, w), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8)
    ax.set_xlabel('Jumlah Cluster (K)', fontsize=12)
    ax.set_ylabel('WCSS', fontsize=12)
    ax.set_title(judul, fontsize=14, fontweight='bold')
    ax.set_xticks(list(k_range))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return wcss_list


# ----------------------------------------------------------
# 10. SCATTER PLOT
# ----------------------------------------------------------
def scatter_cluster(X_2d, labels, centroids_2d, xlabel, ylabel, judul):
    k = len(np.unique(labels))
    warna = ['#e74c3c','#2ecc71','#3498db','#f39c12','#9b59b6']
    fig, ax = plt.subplots(figsize=(9, 6))
    for j in range(k):
        mask = labels == j
        ax.scatter(X_2d[mask,0], X_2d[mask,1],
                   c=warna[j], label=f'Cluster {j+1}', s=80, alpha=0.7, edgecolors='white')
    ax.scatter(centroids_2d[:,0], centroids_2d[:,1],
               c='black', marker='X', s=220, zorder=5, label='Centroid')
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(judul, fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print('Semua fungsi (10 fungsi) berhasil didefinisikan!')


---
# LOAD DATASET

In [ ]:
from google.colab import files
import io

print('Silakan upload file: ATK_-_Inventory_data_For_Ai.xlsx')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_mentah = pd.read_excel(io.BytesIO(uploaded[filename]), skiprows=1)
df_mentah.columns = ['Tanggal_Persetujuan','Tanggal_Permintaan','Jenis_ATK','Divisi','Jumlah','Unit']

print('='*65)
print('  DATA MENTAH (SEBELUM PREPARATION)')
print('='*65)
print(f'  Total baris  : {len(df_mentah):,}')
print(f'  Total kolom  : {df_mentah.shape[1]}')
print(f'  Kolom        : {df_mentah.columns.tolist()}')
print()
display(df_mentah.head())
print()
print('  [Info Tipe Data & Nilai Kosong]')
info_df = pd.DataFrame({
    'Kolom'       : df_mentah.columns,
    'Tipe_Data'   : df_mentah.dtypes.values,
    'Nilai_Kosong': df_mentah.isnull().sum().values,
    'Pct_Kosong'  : (df_mentah.isnull().sum().values / len(df_mentah) * 100).round(2)
})
display(info_df)


---
# 🧹 DATA PREPARATION


In [ ]:
# ============================================================
#  DATA PREPARATION — LANGKAH DEMI LANGKAH
#  Menampilkan tabel SEBELUM dan SESUDAH setiap tahap
# ============================================================

print('='*65)
print('  DATA PREPARATION — RINGKASAN TAHAPAN')
print('='*65)
print()

tahapan_log = []


# ────────────────────────────────────────────────────────────
# TAHAP 1 — Cek & hapus duplikat (dari df_mentah 6 kolom)
# ────────────────────────────────────────────────────────────
print('─'*65)
print('  TAHAP 1 — Verifikasi & hapus duplikat sejati')
print('  Alasan: Cek duplikat dilakukan SEBELUM hapus kolom tanggal')
print('          agar patokannya 6 kolom — termasuk tanggal.')
print('          Baris yang sama Jenis_ATK+Divisi+Jumlah tapi beda')
print('          tanggal bukan duplikat, melainkan transaksi berbeda.')
print('─'*65)

n_sebelum_dup   = len(df_mentah)
n_duplikat      = df_mentah.duplicated().sum()

if n_duplikat > 0:
    print(f'\n  Ditemukan {n_duplikat} duplikat sejati (identik di 6 kolom) -> dihapus')
    print('  [Contoh baris duplikat sejati]')
    display(df_mentah[df_mentah.duplicated(keep=False)].head(6))
    df_step1_dup = df_mentah.drop_duplicates().reset_index(drop=True)
else:
    df_step1_dup = df_mentah.copy()
    print(f'\n  Tidak ada duplikat sejati ditemukan')

comp1 = pd.DataFrame({
    'Kondisi'           : ['Sebelum','Sesudah'],
    'Jumlah Baris'      : [n_sebelum_dup, len(df_step1_dup)],
    'Duplikat Dihapus'  : ['-', str(n_duplikat)],
    'Dasar Perbandingan': ['6 kolom (termasuk tanggal)','6 kolom (termasuk tanggal)']
})
display(comp1)
tahapan_log.append({
    'Tahap'      : '1 — Hapus duplikat sejati',
    'Baris'      : len(df_step1_dup),
    'Kolom'      : df_step1_dup.shape[1],
    'Keterangan' : f'{n_duplikat} duplikat identik (6 kolom) dihapus'
})


# ────────────────────────────────────────────────────────────
# TAHAP 2 — Hapus kolom tidak relevan (Tanggal & Unit)
# ────────────────────────────────────────────────────────────
print()
print('─'*65)
print('  TAHAP 2 — Hapus kolom tidak relevan')
print('  Alasan: Setelah duplikat dicek, baru kolom tanggal & Unit')
print('          dihapus karena tidak digunakan dalam model clustering.')
print('─'*65)

kolom_sebelum = df_step1_dup.columns.tolist()
df_step2 = df_step1_dup.drop(columns=['Tanggal_Persetujuan','Tanggal_Permintaan','Unit'])
kolom_sesudah = df_step2.columns.tolist()

comp2 = pd.DataFrame({
    'Kondisi'      : ['Sebelum','Sesudah'],
    'Jumlah Kolom' : [len(kolom_sebelum), len(kolom_sesudah)],
    'Kolom'        : [str(kolom_sebelum), str(kolom_sesudah)]
})
print()
display(comp2)
print(f'  Kolom dihapus : {set(kolom_sebelum) - set(kolom_sesudah)}')
tahapan_log.append({
    'Tahap'      : '2 — Hapus kolom tidak relevan',
    'Baris'      : len(df_step2),
    'Kolom'      : df_step2.shape[1],
    'Keterangan' : 'Hapus Tanggal_Persetujuan, Tanggal_Permintaan, Unit'
})


# ────────────────────────────────────────────────────────────
# TAHAP 3 — Konversi tipe data Jumlah ke numerik
# ────────────────────────────────────────────────────────────
print()
print('─'*65)
print('  TAHAP 3 — Konversi tipe data kolom Jumlah ke numerik')
print('  Alasan: Kolom Jumlah bisa bertipe object karena header Excel.')
print('          Nilai tidak bisa dikonversi otomatis menjadi NaN.')
print('─'*65)

tipe_sebelum = str(df_step2['Jumlah'].dtype)
n_nonnumeric = pd.to_numeric(df_step2['Jumlah'], errors='coerce').isnull().sum() - df_step2['Jumlah'].isnull().sum()

df_step3 = df_step2.copy()
df_step3['Jumlah'] = pd.to_numeric(df_step3['Jumlah'], errors='coerce')
tipe_sesudah = str(df_step3['Jumlah'].dtype)

comp3 = pd.DataFrame({
    'Kondisi'                 : ['Sebelum','Sesudah'],
    'Tipe Jumlah'             : [tipe_sebelum, tipe_sesudah],
    'NaN di Jumlah'           : [df_step2['Jumlah'].isnull().sum(), df_step3['Jumlah'].isnull().sum()],
    'Nilai Non-Numerik -> NaN': [n_nonnumeric, 0]
})
print()
display(comp3)
tahapan_log.append({
    'Tahap'      : '3 — Konversi tipe numerik',
    'Baris'      : len(df_step3),
    'Kolom'      : df_step3.shape[1],
    'Keterangan' : f'Jumlah: {tipe_sebelum} -> {tipe_sesudah}, {n_nonnumeric} nilai non-numerik jadi NaN'
})


# ────────────────────────────────────────────────────────────
# TAHAP 4 — Hapus baris dengan nilai kosong (NaN)
# ────────────────────────────────────────────────────────────
print()
print('─'*65)
print('  TAHAP 4 — Hapus baris dengan nilai kosong (NaN)')
print('  Alasan: Baris tanpa Divisi, Jenis_ATK, atau Jumlah tidak')
print('          bisa digunakan dalam proses agregasi dan clustering.')
print('─'*65)

n_sebelum4         = len(df_step3)
nan_per_kolom      = df_step3.isnull().sum()

df_step4 = df_step3.dropna(subset=['Jenis_ATK','Divisi','Jumlah']).reset_index(drop=True)
n_sesudah4 = len(df_step4)
n_dihapus4 = n_sebelum4 - n_sesudah4

print()
print('  [NaN per kolom sebelum dropna]')
display(pd.DataFrame({
    'Kolom'      : nan_per_kolom.index,
    'Jumlah NaN' : nan_per_kolom.values,
    'Pct (%)'    : (nan_per_kolom.values / n_sebelum4 * 100).round(2)
}))

comp4 = pd.DataFrame({
    'Kondisi'        : ['Sebelum dropna','Sesudah dropna'],
    'Jumlah Baris'   : [n_sebelum4, n_sesudah4],
    'Baris Dihapus'  : ['-', str(n_dihapus4)],
    'NaN di Divisi'  : [nan_per_kolom.get('Divisi', 0), 0],
    'NaN di Jumlah'  : [nan_per_kolom.get('Jumlah', 0), 0]
})
print()
print('  [Perbandingan Sebelum vs Sesudah]')
display(comp4)
tahapan_log.append({
    'Tahap'      : '4 — Hapus baris NaN',
    'Baris'      : n_sesudah4,
    'Kolom'      : df_step4.shape[1],
    'Keterangan' : f'{n_dihapus4} baris dihapus (NaN di Jenis_ATK/Divisi/Jumlah)'
})


# ────────────────────────────────────────────────────────────
# TAHAP 5 — Validasi nilai Jumlah (<= 0)
# ────────────────────────────────────────────────────────────
print()
print('─'*65)
print('  TAHAP 5 — Validasi nilai Jumlah (<= 0)')
print('  Alasan: Jumlah ATK tidak mungkin nol atau negatif secara')
print('          logika bisnis. Nilai ini menandakan kesalahan input.')
print('─'*65)

n_invalid  = (df_step4['Jumlah'] <= 0).sum()
n_sebelum5 = len(df_step4)

if n_invalid > 0:
    print(f'\n  [Baris tidak valid (Jumlah <= 0)]')
    display(df_step4[df_step4['Jumlah'] <= 0].head())
    df_step5 = df_step4[df_step4['Jumlah'] > 0].reset_index(drop=True)
else:
    df_step5 = df_step4.copy()
    print(f'\n  Semua nilai Jumlah valid (> 0)')

comp5 = pd.DataFrame({
    'Kondisi'           : ['Sebelum','Sesudah'],
    'Jumlah Baris'      : [n_sebelum5, len(df_step5)],
    'Nilai Jumlah <= 0' : [n_invalid, 0],
    'Min Jumlah'        : [df_step4['Jumlah'].min(), df_step5['Jumlah'].min()],
    'Max Jumlah'        : [df_step4['Jumlah'].max(), df_step5['Jumlah'].max()]
})
display(comp5)
tahapan_log.append({
    'Tahap'      : '5 — Validasi Jumlah > 0',
    'Baris'      : len(df_step5),
    'Kolom'      : df_step5.shape[1],
    'Keterangan' : f'{n_invalid} baris Jumlah<=0 dihapus' if n_invalid else 'Semua Jumlah valid (> 0)'
})


# ────────────────────────────────────────────────────────────
# HASIL AKHIR — df_raw siap digunakan semua model
# ────────────────────────────────────────────────────────────
df_raw = df_step5.copy()

print()
print('='*65)
print('  RINGKASAN DATA PREPARATION — BEFORE vs AFTER')
print('='*65)
print()
display(pd.DataFrame(tahapan_log))

print()
print('  [PERBANDINGAN AKHIR: DATA MENTAH vs DATA SIAP]')
comp_final = pd.DataFrame({
    'Metrik'            : ['Jumlah Baris','Jumlah Kolom','Kolom yang Ada',
                           'Nilai Kosong','Duplikat Sejati','Tipe Jumlah',
                           'Min Jumlah','Max Jumlah'],
    'Data Mentah'       : [
        len(df_mentah),
        df_mentah.shape[1],
        str(df_mentah.columns.tolist()),
        int(df_mentah.isnull().sum().sum()),
        int(df_mentah.duplicated().sum()),
        str(df_mentah['Jumlah'].dtype),
        df_mentah['Jumlah'].min(),
        df_mentah['Jumlah'].max()
    ],
    'Data Siap (df_raw)': [
        len(df_raw),
        df_raw.shape[1],
        str(df_raw.columns.tolist()),
        int(df_raw.isnull().sum().sum()),
        int(df_raw.duplicated().sum()),
        str(df_raw['Jumlah'].dtype),
        df_raw['Jumlah'].min(),
        df_raw['Jumlah'].max()
    ]
})
display(comp_final)

print(f'\n  df_raw siap untuk analisis clustering:')
print(f'  Kolom    : {df_raw.columns.tolist()}')
print(f'  Baris    : {len(df_raw):,}  (total transaksi valid)')
print(f'  Divisi   : {df_raw["Divisi"].nunique()}')
print(f'  Jenis ATK: {df_raw["Jenis_ATK"].nunique()}')
print()
display(df_raw.head())


---
# MODEL 1 — CLUSTERING DIVISI (K-Means)
## Konsumsi Tinggi / Sedang / Rendah

## Langkah 1 — INPUT DATA

In [ ]:
m1_raw = df_raw.groupby('Divisi').agg(
    Volume    = ('Jumlah', 'sum'),
    Frekuensi = ('Jumlah', 'count')
).reset_index()

# Filter: hanya divisi dengan Frekuensi >= 5
m1 = m1_raw[m1_raw['Frekuensi'] >= 5].reset_index(drop=True)

print('='*65)
print('  MODEL 1 - AGREGASI PER DIVISI  (filter Frekuensi >= 5)')
print('='*65)
print(f'  Total divisi sebelum filter : {len(m1_raw)}')
print(f'  Divisi dikeluarkan          : {len(m1_raw)-len(m1)}'
      f'  (Frekuensi < 5)')
print(f'  Divisi yang dianalisis      : {len(m1)}')
print()
print('  [Divisi yang Dikeluarkan (Frekuensi < 5)]')
dikeluarkan_m1 = m1_raw[m1_raw['Frekuensi'] < 5]
display(dikeluarkan_m1)
print()
print('  [Data Setelah Filter]')
display(m1)
print()
display(m1[['Volume','Frekuensi']].describe())


## Langkah 2 — NORMALISASI (RobustScaler)

In [ ]:
m1_norm, m1_stats = normalisasi_robust(m1, ['Volume','Frekuensi'])

print('='*65)
print('  NORMALISASI ROBUST SCALER — MODEL 1')
print("  Rumus: X' = (X - Median) / IQR")
print('='*65)

print('\n  [Parameter RobustScaler]')
df_p = pd.DataFrame(m1_stats).T.reset_index()
df_p.columns = ['Fitur','Q1','Median','Q3','IQR']
display(df_p.round(4))

print('\n  [Contoh Perhitungan Normalisasi]')
for col in ['Volume','Frekuensi']:
    s = m1_stats[col]
    row = m1.iloc[0]
    hasil = (row[col] - s['Median']) / s['IQR']
    print(f"  {col}: ({row[col]:.2f} - {s['Median']:.2f}) / {s['IQR']:.2f} = {hasil:.4f}")

print()
display(m1_norm[['Divisi','Volume','Frekuensi','Volume_norm','Frekuensi_norm']])

X_m1    = m1_norm[['Volume_norm','Frekuensi_norm']].values
nama_m1 = m1_norm['Divisi'].tolist()

## Langkah 3 — K-MEANS (K=2)

In [ ]:
print('='*65)
print('  MODEL 1 — K-MEANS MANUAL (K=2)')
print('='*65)

labels_m1_k2, centroids_m1_k2, history_m1_k2 = kmeans_manual(X_m1, k=2, random_state=42)
print()
tampilkan_iterasi(history_m1_k2, X_m1, ['Volume_norm','Frekuensi_norm'], nama_m1, k=2, max_tampil_awal=3)

## Langkah 4 — EVALUASI (K=2)

In [ ]:
eval_m1_k2 = evaluasi_kmeans(X_m1, labels_m1_k2, centroids_m1_k2, k=2, nama_baris=nama_m1)

## Langkah 5 — K-MEANS (K=3)

In [ ]:
print('='*65)
print('  MODEL 1 — K-MEANS MANUAL (K=3)')
print('='*65)

labels_m1_k3, centroids_m1_k3, history_m1_k3 = kmeans_manual(X_m1, k=3, random_state=42)
print()
tampilkan_iterasi(history_m1_k3, X_m1, ['Volume_norm','Frekuensi_norm'], nama_m1, k=3, max_tampil_awal=3)

In [ ]:
eval_m1_k3 = evaluasi_kmeans(X_m1, labels_m1_k3, centroids_m1_k3, k=3, nama_baris=nama_m1)

## Langkah 6 — PERBANDINGAN K & ELBOW

In [ ]:
tabel_eval_m1 = pd.DataFrame([eval_m1_k2, eval_m1_k3]).set_index('K')
print('='*65)
print('  PERBANDINGAN K — MODEL 1')
print('='*65)
display(tabel_eval_m1.round(4))
print()
_ = elbow_method(X_m1, range(2, 8), judul='Elbow Method — Clustering Divisi (Model 1)')

## Langkah 7 — PENENTUAN K TERBAIK

In [ ]:
best_sil_m1 = tabel_eval_m1['Silhouette'].idxmax()
best_dbi_m1 = tabel_eval_m1['DBI'].idxmin()
k_terbaik_m1 = best_sil_m1

print('='*65)
print('  PENENTUAN K TERBAIK — MODEL 1')
print('='*65)
print(f'  Silhouette tertinggi -> K = {best_sil_m1}  ({tabel_eval_m1.loc[best_sil_m1,"Silhouette"]:.4f})')
print(f'  DBI terendah         -> K = {best_dbi_m1}  ({tabel_eval_m1.loc[best_dbi_m1,"DBI"]:.4f})')
print(f'\n  K Terbaik = {k_terbaik_m1}')

## Langkah 8 — HASIL AKHIR MODEL 1

In [ ]:
labels_m1_final    = labels_m1_k2 if k_terbaik_m1 == 2 else labels_m1_k3
centroids_m1_final = centroids_m1_k2 if k_terbaik_m1 == 2 else centroids_m1_k3

m1_result = m1_norm.copy()
m1_result['Cluster_ID'] = labels_m1_final

cluster_vol = m1_result.groupby('Cluster_ID')['Volume'].mean().sort_values(ascending=False)
if k_terbaik_m1 == 2:
    label_map_m1 = {cluster_vol.index[0]: 'Konsumsi Tinggi', cluster_vol.index[1]: 'Konsumsi Rendah'}
else:
    label_map_m1 = {cluster_vol.index[0]: 'Konsumsi Tinggi',
                    cluster_vol.index[1]: 'Konsumsi Sedang',
                    cluster_vol.index[2]: 'Konsumsi Rendah'}

m1_result['Label']   = m1_result['Cluster_ID'].map(label_map_m1)
m1_result['Cluster'] = m1_result['Cluster_ID'].apply(lambda x: f'C{x+1}')

print('='*65)
print(f'  HASIL AKHIR CLUSTERING DIVISI (K={k_terbaik_m1})')
print('='*65)
display(m1_result[['Divisi','Volume','Frekuensi','Cluster','Label']].sort_values('Label'))

print('\n  [Ringkasan Per Cluster]')
display(m1_result.groupby(['Cluster','Label']).agg(
    Jumlah_Divisi  = ('Divisi','count'),
    Rata_Volume    = ('Volume','mean'),
    Rata_Frekuensi = ('Frekuensi','mean')
).reset_index().round(2))

scatter_cluster(X_m1, labels_m1_final, centroids_m1_final,
    'Volume (RobustScaled)', 'Frekuensi (RobustScaled)',
    f'Scatter Plot Clustering Divisi — K={k_terbaik_m1} (Model 1)')

---
# MODEL 2 — CLUSTERING PRODUK ATK (MOVING)

## Langkah 1 — INPUT DATA

In [ ]:
m2_raw = df_raw.groupby('Jenis_ATK').agg(
    Total_Jumlah  = ('Jumlah', 'sum'),
    Frekuensi     = ('Jumlah', 'count'),
    Rata_per_Req  = ('Jumlah', 'mean'),
    Jumlah_Divisi = ('Divisi',  'nunique')
).reset_index()

# Filter: hanya produk dengan Frekuensi >= 3
m2 = m2_raw[m2_raw['Frekuensi'] >= 3].reset_index(drop=True)

print('='*65)
print('  MODEL 2 - AGREGASI PER JENIS ATK  (filter Frekuensi >= 3)')
print('='*65)
print(f'  Total produk sebelum filter : {len(m2_raw)}')
print(f'  Produk dikeluarkan          : {len(m2_raw)-len(m2)}'
      f'  (Frekuensi < 3)')
print(f'  Produk yang dianalisis      : {len(m2)}')
print()
print('  [Produk yang Dikeluarkan (Frekuensi < 3)]')
dikeluarkan_m2 = m2_raw[m2_raw['Frekuensi'] < 3]
display(dikeluarkan_m2)
print()
print('  [Data Setelah Filter]')
display(m2)
print()
display(m2[['Total_Jumlah','Frekuensi','Rata_per_Req','Jumlah_Divisi']].describe())


## Langkah 2 — NORMALISASI (RobustScaler)

In [ ]:
fitur_m2 = ['Total_Jumlah','Frekuensi','Rata_per_Req','Jumlah_Divisi']
m2_norm, m2_stats = normalisasi_robust(m2, fitur_m2)

print('='*65)
print('  NORMALISASI ROBUST SCALER — MODEL 2')
print('='*65)
df_p2 = pd.DataFrame(m2_stats).T.reset_index()
df_p2.columns = ['Fitur','Q1','Median','Q3','IQR']
display(df_p2.round(4))

print('\n  [Contoh Perhitungan Normalisasi — Total_Jumlah]')
s = m2_stats['Total_Jumlah']
for _, row in m2.head(3).iterrows():
    hasil = (row['Total_Jumlah'] - s['Median']) / s['IQR']
    print(f"  {row['Jenis_ATK']:25s}: ({row['Total_Jumlah']:.2f} - {s['Median']:.2f}) / {s['IQR']:.2f} = {hasil:.4f}")

norm_cols_m2 = [f+'_norm' for f in fitur_m2]
print()
display(m2_norm[['Jenis_ATK'] + fitur_m2 + norm_cols_m2])

X_m2    = m2_norm[norm_cols_m2].values
nama_m2 = m2_norm['Jenis_ATK'].tolist()

## Langkah 3 — K-MEANS (K=2)

In [ ]:
print('='*65)
print('  MODEL 2 — K-MEANS MANUAL (K=2)')
print('='*65)

labels_m2_k2, centroids_m2_k2, history_m2_k2 = kmeans_manual(X_m2, k=2, random_state=42)
print()
tampilkan_iterasi(history_m2_k2, X_m2, [f[:13] for f in norm_cols_m2], nama_m2, k=2, max_tampil_awal=3)

## Langkah 4 — EVALUASI (K=2)

In [ ]:
eval_m2_k2 = evaluasi_kmeans(X_m2, labels_m2_k2, centroids_m2_k2, k=2, nama_baris=nama_m2)

## Langkah 5 — K-MEANS (K=3)

In [ ]:
print('='*65)
print('  MODEL 2 — K-MEANS MANUAL (K=3)')
print('='*65)

labels_m2_k3, centroids_m2_k3, history_m2_k3 = kmeans_manual(X_m2, k=3, random_state=42)
print()
tampilkan_iterasi(history_m2_k3, X_m2, [f[:13] for f in norm_cols_m2], nama_m2, k=3, max_tampil_awal=3)

In [ ]:
eval_m2_k3 = evaluasi_kmeans(X_m2, labels_m2_k3, centroids_m2_k3, k=3, nama_baris=nama_m2)

## Langkah 6 — PERBANDINGAN K & ELBOW

In [ ]:
tabel_eval_m2 = pd.DataFrame([eval_m2_k2, eval_m2_k3]).set_index('K')
print('='*65)
print('  PERBANDINGAN K — MODEL 2')
print('='*65)
display(tabel_eval_m2.round(4))
print()
_ = elbow_method(X_m2, range(2, 8), judul='Elbow Method — Clustering Produk ATK (Model 2)')

## Langkah 7 — PENENTUAN K TERBAIK

In [ ]:
best_sil_m2 = tabel_eval_m2['Silhouette'].idxmax()
best_dbi_m2 = tabel_eval_m2['DBI'].idxmin()
k_terbaik_m2 = best_sil_m2

print('='*65)
print('  PENENTUAN K TERBAIK — MODEL 2')
print('='*65)
print(f'  Silhouette tertinggi -> K = {best_sil_m2}  ({tabel_eval_m2.loc[best_sil_m2,"Silhouette"]:.4f})')
print(f'  DBI terendah         -> K = {best_dbi_m2}  ({tabel_eval_m2.loc[best_dbi_m2,"DBI"]:.4f})')
print(f'\n  K Terbaik = {k_terbaik_m2}')

## Langkah 8 — HASIL AKHIR MODEL 2

In [ ]:
labels_m2_final    = labels_m2_k2 if k_terbaik_m2 == 2 else labels_m2_k3
centroids_m2_final = centroids_m2_k2 if k_terbaik_m2 == 2 else centroids_m2_k3

m2_result = m2_norm.copy()
m2_result['Cluster_ID'] = labels_m2_final
cluster_freq = m2_result.groupby('Cluster_ID')['Frekuensi'].mean().sort_values(ascending=False)

if k_terbaik_m2 == 2:
    label_map_m2 = {cluster_freq.index[0]: 'Fast Moving', cluster_freq.index[1]: 'Slow Moving'}
else:
    label_map_m2 = {cluster_freq.index[0]: 'Fast Moving',
                    cluster_freq.index[1]: 'Medium Moving',
                    cluster_freq.index[2]: 'Slow Moving'}

m2_result['Moving_Label'] = m2_result['Cluster_ID'].map(label_map_m2)
m2_result['Cluster']      = m2_result['Cluster_ID'].apply(lambda x: f'C{x+1}')

print('='*65)
print(f'  HASIL AKHIR CLUSTERING PRODUK ATK (K={k_terbaik_m2})')
print('='*65)
display(m2_result[['Jenis_ATK','Total_Jumlah','Frekuensi','Rata_per_Req',
                   'Jumlah_Divisi','Cluster','Moving_Label']].sort_values('Moving_Label'))
print()
display(m2_result.groupby(['Cluster','Moving_Label']).agg(
    Jumlah_Produk  = ('Jenis_ATK','count'),
    Rata_Frekuensi = ('Frekuensi','mean'),
    Rata_TotalJml  = ('Total_Jumlah','mean')
).reset_index().round(2))

scatter_cluster(X_m2[:, :2], labels_m2_final, centroids_m2_final[:, :2],
    'Total_Jumlah (RobustScaled)', 'Frekuensi (RobustScaled)',
    f'Scatter Plot Clustering Produk ATK — K={k_terbaik_m2} (Model 2)')

---
# MODEL 3 — PRIORITAS DIVISI-ITEM ATK

## Langkah 1 — INPUT DATA

In [ ]:
m3_base_raw = df_raw.groupby(['Divisi','Jenis_ATK']).agg(
    Total_Jumlah = ('Jumlah', 'sum'),
    Frekuensi    = ('Jumlah', 'count')
).reset_index()

# Filter: hanya pasangan dengan Frekuensi >= 2
m3_base = m3_base_raw[m3_base_raw['Frekuensi'] >= 2].reset_index(drop=True)

total_per_item   = df_raw.groupby('Jenis_ATK')['Jumlah'].sum().rename('Total_Item_Semua').reset_index()
total_per_divisi = df_raw.groupby('Divisi')['Jumlah'].sum().rename('Total_Divisi_Semua').reset_index()

m3 = (m3_base
      .merge(total_per_item, on='Jenis_ATK')
      .merge(total_per_divisi, on='Divisi'))

m3['Share_Divisi_Terhadap_Item'] = m3['Total_Jumlah'] / m3['Total_Item_Semua']
m3['Share_Item_Dalam_Divisi']    = m3['Total_Jumlah'] / m3['Total_Divisi_Semua']
m3['Frek_Rank']                  = m3['Frekuensi'].rank(pct=True)
m3['Total_Rank']                 = m3['Total_Jumlah'].rank(pct=True)
m3['Skor_Ketergantungan'] = (
    0.35 * m3['Share_Divisi_Terhadap_Item'] +
    0.35 * m3['Share_Item_Dalam_Divisi']   +
    0.20 * m3['Frek_Rank']                 +
    0.10 * m3['Total_Rank']
)

fitur_m3 = ['Total_Jumlah','Frekuensi','Share_Divisi_Terhadap_Item',
            'Share_Item_Dalam_Divisi','Skor_Ketergantungan']

print('='*65)
print('  MODEL 3 - AGREGASI DIVISI x JENIS ATK  (filter Frekuensi >= 2)')
print('='*65)
print(f'  Pasangan sebelum filter : {len(m3_base_raw):,}')
print(f'  Pasangan dikeluarkan    : {len(m3_base_raw)-len(m3_base):,}'
      f'  (Frekuensi < 2)')
print(f'  Pasangan yang dianalisis: {len(m3):,}')
print()
display(m3[['Divisi','Jenis_ATK'] + fitur_m3].head(20))
print()
display(m3[fitur_m3].describe())


## Langkah 2 — NORMALISASI (RobustScaler)

In [ ]:
m3_norm, m3_stats = normalisasi_robust(m3, fitur_m3)

print('='*65)
print('  NORMALISASI ROBUST SCALER — MODEL 3')
print('='*65)
df_p3 = pd.DataFrame(m3_stats).T.reset_index()
df_p3.columns = ['Fitur','Q1','Median','Q3','IQR']
display(df_p3.round(6))

norm_cols_m3 = [f+'_norm' for f in fitur_m3]
print()
display(m3_norm[['Divisi','Jenis_ATK'] + norm_cols_m3].head(20))

X_m3   = m3_norm[norm_cols_m3].values
nama_m3 = (m3_norm['Divisi'] + '|' + m3_norm['Jenis_ATK']).tolist()
print(f'\n  Shape: {X_m3.shape}')

## Langkah 3 — K-MEANS (K=2)

In [ ]:
print('='*65)
print(f'  MODEL 3 — K-MEANS MANUAL (K=2)  [{len(X_m3)} pasangan Divisi-Item]')
print('='*65)

labels_m3_k2, centroids_m3_k2, history_m3_k2 = kmeans_manual(X_m3, k=2, random_state=42)
print()
tampilkan_iterasi(history_m3_k2, X_m3, [f[:15] for f in norm_cols_m3], nama_m3, k=2, max_tampil_awal=3)

## Langkah 4 — EVALUASI (K=2)

In [ ]:
eval_m3_k2 = evaluasi_kmeans(X_m3, labels_m3_k2, centroids_m3_k2, k=2, nama_baris=nama_m3)

## Langkah 5 — K-MEANS (K=3)

In [ ]:
print('='*65)
print('  MODEL 3 — K-MEANS MANUAL (K=3)')
print('='*65)

labels_m3_k3, centroids_m3_k3, history_m3_k3 = kmeans_manual(X_m3, k=3, random_state=42)
print()
tampilkan_iterasi(history_m3_k3, X_m3, [f[:15] for f in norm_cols_m3], nama_m3, k=3, max_tampil_awal=3)

In [ ]:
eval_m3_k3 = evaluasi_kmeans(X_m3, labels_m3_k3, centroids_m3_k3, k=3, nama_baris=nama_m3)

## Langkah 6 — PERBANDINGAN K & ELBOW

In [ ]:
tabel_eval_m3 = pd.DataFrame([eval_m3_k2, eval_m3_k3]).set_index('K')
print('='*65)
print('  PERBANDINGAN K — MODEL 3')
print('='*65)
display(tabel_eval_m3.round(4))
print()
_ = elbow_method(X_m3, range(2, 8), judul='Elbow Method — Prioritas Divisi-Item ATK (Model 3)')

## Langkah 7 — PENENTUAN K TERBAIK

In [ ]:
best_sil_m3 = tabel_eval_m3['Silhouette'].idxmax()
best_dbi_m3 = tabel_eval_m3['DBI'].idxmin()
k_terbaik_m3 = best_sil_m3

print('='*65)
print('  PENENTUAN K TERBAIK — MODEL 3')
print('='*65)
print(f'  Silhouette tertinggi -> K = {best_sil_m3}  ({tabel_eval_m3.loc[best_sil_m3,"Silhouette"]:.4f})')
print(f'  DBI terendah         -> K = {best_dbi_m3}  ({tabel_eval_m3.loc[best_dbi_m3,"DBI"]:.4f})')
print(f'\n  K Terbaik = {k_terbaik_m3}')

## Langkah 8 — HASIL AKHIR MODEL 3

In [ ]:
labels_m3_final    = labels_m3_k2 if k_terbaik_m3 == 2 else labels_m3_k3
centroids_m3_final = centroids_m3_k2 if k_terbaik_m3 == 2 else centroids_m3_k3

m3_result = m3_norm.copy()
m3_result['Cluster_ID'] = labels_m3_final
cluster_skor = m3_result.groupby('Cluster_ID')['Skor_Ketergantungan'].mean().sort_values(ascending=False)

if k_terbaik_m3 == 2:
    label_map_m3 = {cluster_skor.index[0]: 'Prioritas Tinggi',
                    cluster_skor.index[1]: 'Prioritas Rendah'}
else:
    label_map_m3 = {cluster_skor.index[0]: 'Prioritas Tinggi',
                    cluster_skor.index[1]: 'Prioritas Sedang',
                    cluster_skor.index[2]: 'Prioritas Rendah'}

m3_result['Prioritas'] = m3_result['Cluster_ID'].map(label_map_m3)
m3_result['Cluster']   = m3_result['Cluster_ID'].apply(lambda x: f'C{x+1}')

print('='*65)
print(f'  HASIL AKHIR PRIORITAS DIVISI-ITEM (K={k_terbaik_m3})')
print('='*65)
display(m3_result[['Divisi','Jenis_ATK','Total_Jumlah','Frekuensi',
                   'Skor_Ketergantungan','Cluster','Prioritas']]
        .sort_values(['Prioritas','Skor_Ketergantungan'], ascending=[True, False])
        .head(50))

print('\n  [Ringkasan Per Cluster]')
display(m3_result.groupby(['Cluster','Prioritas']).agg(
    Jumlah_Pasangan = ('Divisi','count'),
    Rata_Skor       = ('Skor_Ketergantungan','mean'),
    Rata_Frekuensi  = ('Frekuensi','mean')
).reset_index().round(4))

scatter_cluster(X_m3[:, [0, 4]], labels_m3_final, centroids_m3_final[:, [0, 4]],
    'Total_Jumlah (RobustScaled)', 'Skor_Ketergantungan (RobustScaled)',
    f'Scatter Plot Prioritas Divisi-Item ATK — K={k_terbaik_m3} (Model 3)')

---
# RINGKASAN SELURUH MODEL

In [ ]:
print('='*70)
print('  RINGKASAN EVALUASI SEMUA MODEL')
print('='*70)
all_eval = pd.DataFrame([
    {**eval_m1_k2, 'Model': 'Model 1 (Divisi)'},
    {**eval_m1_k3, 'Model': 'Model 1 (Divisi)'},
    {**eval_m2_k2, 'Model': 'Model 2 (Produk)'},
    {**eval_m2_k3, 'Model': 'Model 2 (Produk)'},
    {**eval_m3_k2, 'Model': 'Model 3 (Div-Item)'},
    {**eval_m3_k3, 'Model': 'Model 3 (Div-Item)'},
])[['Model','K','WCSS','Silhouette','DBI']].round(4)
display(all_eval)
print()
print(f'  Model 1 K Terbaik = {k_terbaik_m1}  |  Model 2 K Terbaik = {k_terbaik_m2}  |  Model 3 K Terbaik = {k_terbaik_m3}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models  = ['Model 1\n(Divisi)', 'Model 2\n(Produk)', 'Model 3\n(Div-Item)']
metrics = [
    ('Silhouette Score (tinggi=baik)',
     [eval_m1_k2['Silhouette'], eval_m2_k2['Silhouette'], eval_m3_k2['Silhouette']],
     [eval_m1_k3['Silhouette'], eval_m2_k3['Silhouette'], eval_m3_k3['Silhouette']]),
    ('DBI (rendah=baik)',
     [eval_m1_k2['DBI'], eval_m2_k2['DBI'], eval_m3_k2['DBI']],
     [eval_m1_k3['DBI'], eval_m2_k3['DBI'], eval_m3_k3['DBI']]),
    ('WCSS (rendah=baik)',
     [eval_m1_k2['WCSS'], eval_m2_k2['WCSS'], eval_m3_k2['WCSS']],
     [eval_m1_k3['WCSS'], eval_m2_k3['WCSS'], eval_m3_k3['WCSS']]),
]

x = np.arange(len(models))
w = 0.35
for ax, (title, vals_k2, vals_k3) in zip(axes, metrics):
    b1 = ax.bar(x - w/2, vals_k2, w, label='K=2', color='#3498db', alpha=0.85)
    b2 = ax.bar(x + w/2, vals_k3, w, label='K=3', color='#e74c3c', alpha=0.85)
    ax.bar_label(b1, fmt='%.3f', fontsize=7, padding=2)
    ax.bar_label(b2, fmt='%.3f', fontsize=7, padding=2)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Perbandingan Evaluasi K=2 vs K=3 (RobustScaler) — Semua Model',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Analisis ATK dengan K-Means (RobustScaler) selesai!')

---
# 📥 EXPORT HASIL KE EXCEL

In [ ]:
# ============================================================
#  EXPORT HASIL KE EXCEL (semua tahapan + hasil akhir)
# ============================================================
from google.colab import files as colab_files
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
import io as _io

def style_header(ws, row_num, fill_color='2E75B6'):
    """Beri warna header baris row_num."""
    fill = PatternFill(start_color=fill_color, end_color=fill_color, fill_type='solid')
    font = Font(bold=True, color='FFFFFF')
    for cell in ws[row_num]:
        if cell.value is not None:
            cell.fill = fill
            cell.font = font
            cell.alignment = Alignment(horizontal='center', vertical='center')

def auto_col_width(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = col[0].column_letter
        for cell in col:
            try:
                max_len = max(max_len, len(str(cell.value)))
            except:
                pass
        ws.column_dimensions[col_letter].width = min(max_len + 4, 40)

def df_to_sheet(wb, sheet_name, df, title=None, header_color='2E75B6'):
    ws = wb.create_sheet(sheet_name)
    start_row = 1
    if title:
        ws.cell(1, 1, title)
        ws.cell(1, 1).font = Font(bold=True, size=12)
        start_row = 2
    for r_idx, row in enumerate(dataframe_to_rows(df.round(4), index=True, header=True), start=start_row):
        for c_idx, val in enumerate(row, 1):
            ws.cell(r_idx, c_idx, val)
    style_header(ws, start_row, fill_color=header_color)
    auto_col_width(ws)
    return ws

wb = openpyxl.Workbook()
wb.remove(wb.active)  # hapus sheet default

# ---- Sheet 1: Info Filter -----------------------------------------------
ws0 = wb.create_sheet('0_Filter_Data')
ws0.cell(1,1,'Ringkasan Filter Data').font = Font(bold=True, size=12)
ws0.cell(3,1,'Model'); ws0.cell(3,2,'Kriteria Filter')
ws0.cell(3,3,'Sebelum Filter'); ws0.cell(3,4,'Setelah Filter')
ws0.cell(3,5,'Dikeluarkan')
style_header(ws0, 3)
data_filter = [
    ('Model 1 - Divisi',      'Frekuensi >= 5', len(m1_raw),      len(m1),      len(m1_raw)-len(m1)),
    ('Model 2 - Produk ATK',  'Frekuensi >= 3', len(m2_raw),      len(m2),      len(m2_raw)-len(m2)),
    ('Model 3 - Divisi-Item', 'Frekuensi >= 2', len(m3_base_raw), len(m3_base), len(m3_base_raw)-len(m3_base)),
]
for i, row in enumerate(data_filter, 4):
    for j, val in enumerate(row, 1):
        ws0.cell(i, j, val)
auto_col_width(ws0)

# ---- Sheet 2: Model 1 - Data Normalisasi ---------------------------------
df_m1_export = m1_norm[['Divisi','Volume','Frekuensi','Volume_norm','Frekuensi_norm']].copy()
df_to_sheet(wb, '1_M1_Data_Normalisasi', df_m1_export,
            title='Model 1 - Data Divisi Setelah RobustScaler', header_color='1F7A4D')

# ---- Sheet 3: Model 1 - Iterasi K=Terbaik --------------------------------
labels_m1_use = labels_m1_k2 if k_terbaik_m1==2 else labels_m1_k3
history_m1_use = history_m1_k2 if k_terbaik_m1==2 else history_m1_k3
ws_m1_it = wb.create_sheet('1_M1_Iterasi_KMeans')
ws_m1_it.cell(1,1,f'Model 1 - Iterasi K-Means (K={k_terbaik_m1})').font = Font(bold=True, size=12)
cur_row = 3
for rec in history_m1_use:
    tag = ' [KONVERGEN]' if rec['is_converged'] else ''
    ws_m1_it.cell(cur_row, 1, f'Iterasi {rec["iterasi"]}{tag}').font = Font(bold=True)
    cur_row += 1
    headers = nama_m1 and ['Divisi'] + [f'Jarak_C{j+1}' for j in range(k_terbaik_m1)] + ['Cluster']
    for c_idx, h in enumerate(headers, 1):
        ws_m1_it.cell(cur_row, c_idx, h)
    style_header(ws_m1_it, cur_row)
    cur_row += 1
    for i_row, (nm, dist_row, lbl) in enumerate(zip(nama_m1, rec['distances'], rec['labels'])):
        ws_m1_it.cell(cur_row, 1, nm)
        for j, d in enumerate(dist_row):
            ws_m1_it.cell(cur_row, j+2, round(float(d), 4))
        ws_m1_it.cell(cur_row, k_terbaik_m1+2, f'C{lbl+1}')
        cur_row += 1
    cur_row += 1
auto_col_width(ws_m1_it)

# ---- Sheet 4: Model 1 - Hasil Akhir --------------------------------------
df_m1_hasil = m1_result[['Divisi','Volume','Frekuensi','Cluster','Label']].copy()
df_to_sheet(wb, '1_M1_Hasil_Akhir', df_m1_hasil,
            title=f'Model 1 - Hasil Clustering Divisi (K={k_terbaik_m1})', header_color='1F7A4D')

# ---- Sheet 5: Model 1 - Evaluasi -----------------------------------------
df_eval_m1 = pd.DataFrame([eval_m1_k2, eval_m1_k3])
df_to_sheet(wb, '1_M1_Evaluasi', df_eval_m1,
            title='Model 1 - Tabel Evaluasi (WCSS, Silhouette, DBI)', header_color='1F7A4D')

# ---- Sheet 6: Model 2 - Data Normalisasi ---------------------------------
norm_cols_m2_exp = [f+'_norm' for f in ['Total_Jumlah','Frekuensi','Rata_per_Req','Jumlah_Divisi']]
df_m2_export = m2_norm[['Jenis_ATK','Total_Jumlah','Frekuensi','Rata_per_Req','Jumlah_Divisi'] + norm_cols_m2_exp].copy()
df_to_sheet(wb, '2_M2_Data_Normalisasi', df_m2_export,
            title='Model 2 - Data Produk ATK Setelah RobustScaler', header_color='7030A0')

# ---- Sheet 7: Model 2 - Iterasi K=Terbaik --------------------------------
labels_m2_use   = labels_m2_k2 if k_terbaik_m2==2 else labels_m2_k3
history_m2_use  = history_m2_k2 if k_terbaik_m2==2 else history_m2_k3
ws_m2_it = wb.create_sheet('2_M2_Iterasi_KMeans')
ws_m2_it.cell(1,1,f'Model 2 - Iterasi K-Means (K={k_terbaik_m2})').font = Font(bold=True, size=12)
cur_row = 3
for rec in history_m2_use:
    tag = ' [KONVERGEN]' if rec['is_converged'] else ''
    ws_m2_it.cell(cur_row, 1, f'Iterasi {rec["iterasi"]}{tag}').font = Font(bold=True)
    cur_row += 1
    headers2 = ['Jenis_ATK'] + [f'Jarak_C{j+1}' for j in range(k_terbaik_m2)] + ['Cluster']
    for c_idx, h in enumerate(headers2, 1):
        ws_m2_it.cell(cur_row, c_idx, h)
    style_header(ws_m2_it, cur_row)
    cur_row += 1
    for nm, dist_row, lbl in zip(nama_m2, rec['distances'], rec['labels']):
        ws_m2_it.cell(cur_row, 1, nm)
        for j, d in enumerate(dist_row):
            ws_m2_it.cell(cur_row, j+2, round(float(d), 4))
        ws_m2_it.cell(cur_row, k_terbaik_m2+2, f'C{lbl+1}')
        cur_row += 1
    cur_row += 1
auto_col_width(ws_m2_it)

# ---- Sheet 8: Model 2 - Hasil Akhir --------------------------------------
df_m2_hasil = m2_result[['Jenis_ATK','Total_Jumlah','Frekuensi','Rata_per_Req',
                          'Jumlah_Divisi','Cluster','Moving_Label']].copy()
df_to_sheet(wb, '2_M2_Hasil_Akhir', df_m2_hasil,
            title=f'Model 2 - Hasil Clustering Produk ATK (K={k_terbaik_m2})', header_color='7030A0')

# ---- Sheet 9: Model 2 - Evaluasi -----------------------------------------
df_eval_m2 = pd.DataFrame([eval_m2_k2, eval_m2_k3])
df_to_sheet(wb, '2_M2_Evaluasi', df_eval_m2,
            title='Model 2 - Tabel Evaluasi (WCSS, Silhouette, DBI)', header_color='7030A0')


# ---- Sheet 10: Model 3 - Data Normalisasi --------------------------------
norm_cols_m3_exp = [f+'_norm' for f in fitur_m3]
df_m3_export = m3_norm[['Divisi','Jenis_ATK'] + fitur_m3 + norm_cols_m3_exp].copy()
df_to_sheet(wb, '3_M3_Data_Normalisasi', df_m3_export,
            title='Model 3 - Data Divisi-Item Setelah RobustScaler', header_color='833C00')

# ---- Sheet 11: Model 3 - Iterasi K=Terbaik --------------------------------
labels_m3_use   = labels_m3_k2 if k_terbaik_m3==2 else labels_m3_k3
history_m3_use  = history_m3_k2 if k_terbaik_m3==2 else history_m3_k3
ws_m3_it = wb.create_sheet('3_M3_Iterasi_KMeans')
ws_m3_it.cell(1,1,f'Model 3 - Iterasi K-Means (K={k_terbaik_m3})').font = Font(bold=True, size=12)
cur_row = 3
for rec in history_m3_use:
    tag = ' [KONVERGEN]' if rec['is_converged'] else ''
    ws_m3_it.cell(cur_row, 1, f'Iterasi {rec["iterasi"]}{tag}').font = Font(bold=True)
    cur_row += 1
    headers3 = ['Divisi|Item'] + [f'Jarak_C{j+1}' for j in range(k_terbaik_m3)] + ['Cluster']
    for c_idx, h in enumerate(headers3, 1):
        ws_m3_it.cell(cur_row, c_idx, h)
    style_header(ws_m3_it, cur_row)
    cur_row += 1
    for nm, dist_row, lbl in zip(nama_m3, rec['distances'], rec['labels']):
        ws_m3_it.cell(cur_row, 1, nm)
        for j, d in enumerate(dist_row):
            ws_m3_it.cell(cur_row, j+2, round(float(d), 4))
        ws_m3_it.cell(cur_row, k_terbaik_m3+2, f'C{lbl+1}')
        cur_row += 1
    cur_row += 1
auto_col_width(ws_m3_it)

# ---- Sheet 12: Model 3 - Hasil Akhir -------------------------------------
df_m3_hasil = m3_result[['Divisi','Jenis_ATK','Total_Jumlah','Frekuensi',
                          'Skor_Ketergantungan','Cluster','Prioritas']].copy()
df_to_sheet(wb, '3_M3_Hasil_Akhir', df_m3_hasil,
            title=f'Model 3 - Hasil Prioritas Divisi-Item (K={k_terbaik_m3})', header_color='833C00')

# ---- Sheet 13: Model 3 - Evaluasi ----------------------------------------
df_eval_m3 = pd.DataFrame([eval_m3_k2, eval_m3_k3])
df_to_sheet(wb, '3_M3_Evaluasi', df_eval_m3,
            title='Model 3 - Tabel Evaluasi (WCSS, Silhouette, DBI)', header_color='833C00')

# ---- Sheet 14: Ringkasan Semua Model --------------------------------------
all_eval_df = pd.DataFrame([
    {**eval_m1_k2, 'Model': 'Model 1 (Divisi)'},
    {**eval_m1_k3, 'Model': 'Model 1 (Divisi)'},
    {**eval_m2_k2, 'Model': 'Model 2 (Produk)'},
    {**eval_m2_k3, 'Model': 'Model 2 (Produk)'},
    {**eval_m3_k2, 'Model': 'Model 3 (Div-Item)'},
    {**eval_m3_k3, 'Model': 'Model 3 (Div-Item)'},
])[['Model','K','WCSS','Silhouette','DBI']]
df_to_sheet(wb, '0_Ringkasan_Evaluasi', all_eval_df,
            title='Ringkasan Evaluasi Semua Model', header_color='2E75B6')


# ---- Helper: tulis evaluasi detail ke sheet ---------------------------------
def write_eval_detail_sheet(wb, sheet_name, X, labels, centroids, k, nama_baris, header_color, kolom_fitur=None):
    """Tulis WCSS per titik, Silhouette per titik, DBI ke satu sheet."""
    from sklearn.metrics import silhouette_samples
    ws = wb.create_sheet(sheet_name)
    ws.cell(1,1, f'Evaluasi Detail K={k}').font = Font(bold=True, size=12)
    cur = 3

    # -- WCSS --
    ws.cell(cur, 1, '4.1  WITHIN-CLUSTER SUM OF SQUARES (WCSS)').font = Font(bold=True)
    ws.cell(cur+1, 1, 'WCSS = Sum_cluster Sum_{i in cluster} ||x_i - mu_cluster||^2')
    cur += 3
    # Nama fitur: jika kolom_fitur disediakan, pakai; jika tidak, pakai F1,F2,...
    hdr_wcss = ['Nama', 'Cluster'] + [f'(x-mu)_{f}^2' for f in (kolom_fitur if kolom_fitur else [f'F{fi+1}' for fi in range(X.shape[1])])] + ['||x-mu||^2']
    for ci, h in enumerate(hdr_wcss, 1):
        ws.cell(cur, ci, h)
    style_header(ws, cur, fill_color=header_color)
    cur += 1

    wcss_total = 0
    cluster_ss = {f'C{j+1}': 0 for j in range(k)}
    for j in range(k):
        for idx in np.where(labels == j)[0]:
            diff2 = (X[idx] - centroids[j])**2
            sq    = float(np.sum(diff2))
            cluster_ss[f'C{j+1}'] += sq
            wcss_total += sq
            ws.cell(cur, 1, nama_baris[idx])
            ws.cell(cur, 2, f'C{j+1}')
            for fi, v in enumerate(diff2):
                ws.cell(cur, 3+fi, round(float(v), 6))
            ws.cell(cur, 3+X.shape[1], round(sq, 6))
            cur += 1

    cur += 1
    ws.cell(cur, 1, 'SS per Cluster').font = Font(bold=True)
    cur += 1
    ws.cell(cur, 1, 'Cluster'); ws.cell(cur, 2, 'n_anggota'); ws.cell(cur, 3, 'SS_Cluster')
    style_header(ws, cur, fill_color=header_color); cur += 1
    for j in range(k):
        n_j = int(np.sum(labels == j))
        ws.cell(cur, 1, f'C{j+1}'); ws.cell(cur, 2, n_j)
        ws.cell(cur, 3, round(cluster_ss[f'C{j+1}'], 6)); cur += 1
    ws.cell(cur, 1, 'WCSS Total'); ws.cell(cur, 3, round(wcss_total, 6))
    ws.cell(cur, 1).font = Font(bold=True); cur += 3

    # -- Silhouette --
    ws.cell(cur, 1, '4.2  SILHOUETTE SCORE').font = Font(bold=True)
    ws.cell(cur+1, 1, 'a(i)=rata2 jarak intra-cluster   b(i)=rata2 jarak ke cluster terdekat   s(i)=(b-a)/max(a,b)')
    cur += 3
    hdr_sil = ['Nama', 'Cluster', 'a(i)', 'b(i)', 's(i)']
    for ci, h in enumerate(hdr_sil, 1):
        ws.cell(cur, ci, h)
    style_header(ws, cur, fill_color=header_color); cur += 1

    sil_vals      = silhouette_samples(X, labels)
    unique_labels = np.unique(labels)
    a_vals = np.zeros(len(X)); b_vals = np.zeros(len(X))
    for i in range(len(X)):
        ci   = labels[i]
        same = np.where(labels == ci)[0]; same = same[same != i]
        a_vals[i] = float(np.mean([np.sqrt(np.sum((X[i]-X[j])**2)) for j in same])) if len(same) > 0 else 0.0
        b_cands = []
        for cj in unique_labels:
            if cj == ci: continue
            other = np.where(labels == cj)[0]
            b_cands.append(float(np.mean([np.sqrt(np.sum((X[i]-X[j])**2)) for j in other])))
        b_vals[i] = min(b_cands) if b_cands else 0.0

    for i in range(len(X)):
        ws.cell(cur, 1, nama_baris[i])
        ws.cell(cur, 2, f'C{labels[i]+1}')
        ws.cell(cur, 3, round(a_vals[i], 4))
        ws.cell(cur, 4, round(b_vals[i], 4))
        ws.cell(cur, 5, round(float(sil_vals[i]), 4))
        cur += 1

    cur += 1
    ws.cell(cur, 1, 'Rata-rata s(i) per Cluster').font = Font(bold=True); cur += 1
    ws.cell(cur, 1, 'Cluster'); ws.cell(cur, 2, 'mean s(i)'); ws.cell(cur, 3, 'n')
    style_header(ws, cur, fill_color=header_color); cur += 1
    for lbl in unique_labels:
        mask = labels == lbl
        ws.cell(cur, 1, f'C{lbl+1}')
        ws.cell(cur, 2, round(float(sil_vals[mask].mean()), 4))
        ws.cell(cur, 3, int(np.sum(mask))); cur += 1
    sil_mean = float(sil_vals.mean())
    ws.cell(cur, 1, 'Silhouette Score'); ws.cell(cur, 2, round(sil_mean, 4))
    ws.cell(cur, 1).font = Font(bold=True); cur += 3

    # -- DBI --
    ws.cell(cur, 1, '4.3  DAVIES-BOULDIN INDEX (DBI)').font = Font(bold=True)
    ws.cell(cur+1, 1, 's_i=intra-scatter   d_ij=dist(centroid_i, centroid_j)   R_ij=(s_i+s_j)/d_ij   D_i=max(R_ij)')
    cur += 3

    s = np.zeros(k)
    for j in range(k):
        mask = labels == j
        if np.any(mask):
            s[j] = float(np.mean([np.sqrt(np.sum((X[i]-centroids[j])**2)) for i in np.where(mask)[0]]))

    ws.cell(cur, 1, 's_i (Intra-Cluster Scatter)').font = Font(bold=True); cur += 1
    ws.cell(cur, 1, 'Cluster'); ws.cell(cur, 2, 's_i')
    style_header(ws, cur, fill_color=header_color); cur += 1
    for j in range(k):
        ws.cell(cur, 1, f'C{j+1}'); ws.cell(cur, 2, round(s[j], 4)); cur += 1
    cur += 1

    d = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            d[i,j] = float(np.sqrt(np.sum((centroids[i]-centroids[j])**2)))
    ws.cell(cur, 1, 'd_ij (Jarak Antar Centroid)').font = Font(bold=True); cur += 1
    ws.cell(cur, 1, '')
    for j in range(k):
        ws.cell(cur, j+2, f'C{j+1}')
    style_header(ws, cur, fill_color=header_color); cur += 1
    for i in range(k):
        ws.cell(cur, 1, f'C{i+1}')
        for j in range(k):
            ws.cell(cur, j+2, round(d[i,j], 4))
        cur += 1
    cur += 1

    R = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if i != j and d[i,j] > 0:
                R[i,j] = (s[i]+s[j]) / d[i,j]
    ws.cell(cur, 1, 'R_ij = (s_i + s_j) / d_ij').font = Font(bold=True); cur += 1
    ws.cell(cur, 1, '')
    for j in range(k):
        ws.cell(cur, j+2, f'C{j+1}')
    style_header(ws, cur, fill_color=header_color); cur += 1
    for i in range(k):
        ws.cell(cur, 1, f'C{i+1}')
        for j in range(k):
            ws.cell(cur, j+2, round(R[i,j], 4))
        cur += 1
    cur += 1

    D = np.array([max(R[i,j] for j in range(k) if j!=i) for i in range(k)])
    ws.cell(cur, 1, 'D_i = max_{j!=i} R_ij').font = Font(bold=True); cur += 1
    ws.cell(cur, 1, 'Cluster'); ws.cell(cur, 2, 'D_i')
    style_header(ws, cur, fill_color=header_color); cur += 1
    for i in range(k):
        ws.cell(cur, 1, f'C{i+1}'); ws.cell(cur, 2, round(D[i], 4)); cur += 1
    dbi = float(np.mean(D))
    ws.cell(cur, 1, 'DBI'); ws.cell(cur, 2, round(dbi, 4))
    ws.cell(cur, 1).font = Font(bold=True); cur += 2

    auto_col_width(ws)
    return ws


# ---- Sheet Evaluasi Detail M1 — K=2 dan K=3 ----------------------------
write_eval_detail_sheet(wb, '1_M1_Eval_Detail_K2',
    X_m1, labels_m1_k2, centroids_m1_k2, 2, nama_m1, '1F7A4D', ['Volume_norm','Frekuensi_norm'])
write_eval_detail_sheet(wb, '1_M1_Eval_Detail_K3',
    X_m1, labels_m1_k3, centroids_m1_k3, 3, nama_m1, '1F7A4D', ['Volume_norm','Frekuensi_norm'])

# ---- Sheet Evaluasi Detail M2 — K=2 dan K=3 ----------------------------
write_eval_detail_sheet(wb, '2_M2_Eval_Detail_K2',
    X_m2, labels_m2_k2, centroids_m2_k2, 2, nama_m2, '7030A0', ['Total_Jumlah_norm','Frekuensi_norm','Rata_per_Req_norm','Jumlah_Divisi_norm'])
write_eval_detail_sheet(wb, '2_M2_Eval_Detail_K3',
    X_m2, labels_m2_k3, centroids_m2_k3, 3, nama_m2, '7030A0', ['Total_Jumlah_norm','Frekuensi_norm','Rata_per_Req_norm','Jumlah_Divisi_norm'])

# ---- Sheet Evaluasi Detail M3 — K=2 dan K=3 ----------------------------
write_eval_detail_sheet(wb, '3_M3_Eval_Detail_K2',
    X_m3, labels_m3_k2, centroids_m3_k2, 2, nama_m3, '833C00', ['Total_Jumlah_norm','Frekuensi_norm','Share_D_norm','Share_I_norm','Skor_norm'])
write_eval_detail_sheet(wb, '3_M3_Eval_Detail_K3',
    X_m3, labels_m3_k3, centroids_m3_k3, 3, nama_m3, '833C00', ['Total_Jumlah_norm','Frekuensi_norm','Share_D_norm','Share_I_norm','Skor_norm'])

# ---- Simpan & Download ---------------------------------------------------
output_fname = 'Hasil_Analisis_ATK_KMeans.xlsx'
buf = _io.BytesIO()
wb.save(buf)
buf.seek(0)
with open(output_fname, 'wb') as f:
    f.write(buf.read())

colab_files.download(output_fname)
print(f'File {output_fname} siap didownload!')
print(f'File {output_fname} siap didownload!')
print()
print('Sheet yang tersedia:')
groups = {
    '0 Info'         : [s for s in wb.sheetnames if s.startswith('0_')],
    '1 Model 1'      : [s for s in wb.sheetnames if s.startswith('1_')],
    '2 Model 2'      : [s for s in wb.sheetnames if s.startswith('2_')],
    '3 Model 3'      : [s for s in wb.sheetnames if s.startswith('3_')],
}
for grp, sheets in groups.items():
    print(f'  [{grp}]')
    for s in sheets:
        tag = ' ★ K terpilih' if ('Eval_Detail' in s and (
            (s.endswith(f'K{k_terbaik_m1}') and '1_M1' in s) or
            (s.endswith(f'K{k_terbaik_m2}') and '2_M2' in s) or
            (s.endswith(f'K{k_terbaik_m3}') and '3_M3' in s)
        )) else ''
        print(f'    {s}{tag}')
